# Evaluating an agent with LangChain and Langfuse

An agent chooses its own steps, so we evaluate what it produced, not only what it said.

We run two experiments. Each one compares the same two models:

1. **without tools**, scored on the answer,
2. **with tools**, scored on the answer and on the file the agent wrote.

In [ ]:
%pip install -q "langchain>=1.0,<2" "langchain-openrouter>=0.1,<1" "langfuse>=4,<5" "pandas>=2.2,<4" "python-dotenv>=1.0,<2"

In [7]:
import json
import os
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openrouter import ChatOpenRouter
from langfuse import Evaluation, get_client
from langfuse.langchain import CallbackHandler

load_dotenv()

langfuse = get_client()
langfuse_handler = CallbackHandler()
print("Langfuse connected:", langfuse.auth_check())

APP_TITLE = "PyData Amsterdam LLM Evaluation Tutorial"
model_names = [
    os.getenv("OPENROUTER_MODEL", "openai/gpt-4.1-mini"),
    os.getenv("OPENROUTER_COMPARISON_MODEL", "google/gemini-2.5-flash"),
]

models = {
    name: ChatOpenRouter(model=name, app_title=APP_TITLE) for name in model_names
}
model_names

Langfuse connected: True


['openai/gpt-4.1-mini', 'google/gemini-2.5-flash']

## 1. The data

`dataset.json` holds the conference programme and the test examples.

In [8]:
dataset_candidates = [
    Path("dataset.json"),
    Path("02_agent/dataset.json"),
    Path("examples/02_agent/dataset.json"),
]
data = json.loads(next(p for p in dataset_candidates if p.exists()).read_text())

talks = data["talks"]
examples = data["examples"]
pd.DataFrame(talks).head()

,talk_id,title,speaker,topic,day,start,end,room
0,t01,Transformers Without the Hype,Lena Vos,nlp,Thursday,10:30,11:10,Zaal 1
1,t02,Shipping Models on Friday,Ravi Menon,mlops,Thursday,10:30,11:10,Zaal 2
2,t03,Evaluating LLM Applications,Mira Okafor,nlp,Thursday,11:20,12:00,Zaal 1
3,t04,Charts That Explain Themselves,Joost Bakker,viz,Thursday,11:20,12:00,Zaal 2
4,t05,Streaming with Arrow,Ana Ferreira,data-eng,Thursday,13:00,13:40,Zaal 1


## 2. Prompts in Langfuse

Both system prompts are stored in Langfuse, so they can be changed without editing this notebook.

The first prompt takes the programme as a `{{programme}}` variable. The second one names the tools instead, so the agent knows what it can call.

In [3]:
SIMPLE_PROMPT = "pydata-schedule-assistant"
TOOLS_PROMPT = "pydata-schedule-assistant-tools"

ANSWER_FORMAT = (
    "List each talk on its own line as:\n<title> - <speaker> - <day> <start>-<end>"
)

langfuse.create_prompt(
    name=SIMPLE_PROMPT,
    type="text",
    prompt=(
        "You are a PyData Amsterdam schedule assistant.\n\n"
        "Programme:\n{{programme}}\n\n" + ANSWER_FORMAT
    ),
    labels=["production"],
)

langfuse.create_prompt(
    name=TOOLS_PROMPT,
    type="text",
    prompt=(
        "You are a PyData Amsterdam schedule assistant.\n\n"
        "You have two tools:\n"
        "- search_talks(topic): the talks for one topic, "
        "which is nlp, mlops, viz or data-eng.\n"
        "- save_favorites(talk_ids): save the chosen talks to a file.\n\n"
        "Always search before you answer. "
        "Then save every talk you listed with save_favorites. "
        "Never ask the attendee to confirm, just save them.\n\n" + ANSWER_FORMAT
    ),
    labels=["production"],
)

In [9]:
programme = "\n".join(
    f"{t['talk_id']}: {t['title']} by {t['speaker']} "
    f"({t['topic']}, {t['day']} {t['start']}-{t['end']}, {t['room']})"
    for t in talks
)

simple_prompt = langfuse.get_prompt(SIMPLE_PROMPT).compile(programme=programme)
tools_prompt = langfuse.get_prompt(TOOLS_PROMPT).compile()

print(tools_prompt)

You are a PyData Amsterdam schedule assistant.

You have two tools:
- search_talks(topic): the talks for one topic, which is nlp, mlops, viz or data-eng.
- save_favorites(talk_ids): save the chosen talks to a file.

Always search before you answer, then save the talks the attendee is interested in.

List each talk on its own line as:
<title> - <speaker> - <day> <start>-<end>


## 3. Tools and agents

`save_favorites` writes a CSV file, which gives us an output we can check on disk instead of only reading text.

In [10]:
OUTPUT_FILE = Path("output/favorites.csv")


@tool
def search_talks(topic: str) -> list[dict]:
    """Find talks about a topic: nlp, mlops, viz or data-eng."""
    return [talk for talk in talks if talk["topic"] == topic]


@tool
def save_favorites(talk_ids: list[str]) -> str:
    """Save the chosen talks to a CSV file."""
    rows = [talk for talk in talks if talk["talk_id"] in talk_ids]
    OUTPUT_FILE.parent.mkdir(exist_ok=True)
    pd.DataFrame(rows).to_csv(OUTPUT_FILE, index=False)
    return f"Saved {len(rows)} talks."


TOOLS = [search_talks, save_favorites]

simple_agents = {
    name: create_agent(model, system_prompt=simple_prompt)
    for name, model in models.items()
}
tool_agents = {
    name: create_agent(model, tools=TOOLS, system_prompt=tools_prompt)
    for name, model in models.items()
}

In [11]:
result = tool_agents[model_names[0]].invoke(
    {"messages": [{"role": "user", "content": "I am interested in NLP talks on Thursday."}]},
    config={"callbacks": [langfuse_handler]},
)
print(result["messages"][-1].content)

pd.read_csv(OUTPUT_FILE) if OUTPUT_FILE.exists() else "The agent saved nothing."

I have saved the NLP talks on Thursday you are interested in:
- Transformers Without the Hype - Lena Vos - Thursday 10:30-11:10
- Evaluating LLM Applications - Mira Okafor - Thursday 11:20-12:00
- Multilingual Embeddings in Practice - Yusuf Aydin - Thursday 13:00-13:40

Let me know if you want info on talks in other topics or days.


,talk_id,title,speaker,topic,day,start,end,room
0,t01,Transformers Without the Hype,Lena Vos,nlp,Thursday,10:30,11:10,Zaal 1
1,t03,Evaluating LLM Applications,Mira Okafor,nlp,Thursday,11:20,12:00,Zaal 1
2,t06,Multilingual Embeddings in Practice,Yusuf Aydin,nlp,Thursday,13:00,13:40,Zaal 2


## 4. Task and evaluators

The task returns the answer text and the talks that reached the file. `finds_expected_talks` reads the first, `saves_expected_talks` reads the second.

A talk counts as found only when its title, speaker, day and start time all appear in the answer.

In [12]:
def create_task(agent):
    def task(*, item, **kwargs) -> dict:
        # Delete the file the previous example wrote, so we only read this run.
        OUTPUT_FILE.unlink(missing_ok=True)

        result = agent.invoke(
            {"messages": [{"role": "user", "content": item.input["request"]}]},
            config={"callbacks": [langfuse_handler]},
        )
        saved = pd.read_csv(OUTPUT_FILE)["title"].tolist() if OUTPUT_FILE.exists() else []
        return {"answer": result["messages"][-1].content, "saved_titles": saved}

    return task


def finds_expected_talks(*, output, expected_output, **kwargs) -> Evaluation:
    answer = output["answer"].casefold()
    expected = expected_output["talks"]

    found = 0
    problems = []
    for talk in expected:
        missing = [field for field, value in talk.items() if value.casefold() not in answer]
        if missing:
            problems.append(f"{talk['title']}: no {', '.join(missing)}")
        else:
            found += 1

    return Evaluation(
        name="finds_expected_talks",
        value=found / len(expected),
        comment="; ".join(problems) or "All talks found.",
    )


def saves_expected_talks(*, output, expected_output, **kwargs) -> Evaluation:
    expected = {talk["title"] for talk in expected_output["talks"]}
    saved = set(output["saved_titles"])

    problems = []
    if expected - saved:
        problems.append(f"not saved: {', '.join(sorted(expected - saved))}")
    if saved - expected:
        problems.append(f"saved but not expected: {', '.join(sorted(saved - expected))}")

    return Evaluation(
        name="saves_expected_talks",
        value=saved == expected,
        comment="; ".join(problems) or "The file matches.",
    )

## 5. The dataset in Langfuse

In [13]:
timestamp = datetime.now(UTC).strftime("%Y%m%d-%H%M%S")
dataset_name = f"pydata-agent-{timestamp}"

langfuse.create_dataset(name=dataset_name, description="Conference schedule requests.")
for example in examples:
    langfuse.create_dataset_item(dataset_name=dataset_name, **example)

dataset = langfuse.get_dataset(dataset_name)
print(f"{dataset.name} has {len(dataset.items)} items")

## 6. Reading the results

`compare` puts the experiment results into one table. Next to every score it shows the comment the evaluator wrote, so a low score says why it is low without opening a trace.

In [ ]:
def compare(results: dict) -> pd.DataFrame:
    frames = {}
    for name, result in results.items():
        rows = []
        for item_result in result.item_results:
            row = {"request": item_result.item.input["request"]}
            for evaluation in item_result.evaluations:
                row[evaluation.name] = evaluation.value
                row[f"{evaluation.name}: why"] = evaluation.comment
            rows.append(row)
        frames[name] = pd.DataFrame(rows)
    return pd.concat(frames, names=["experiment"])

## 7. Experiment 1: answering from the programme in the prompt

The whole programme sits in the system prompt. Each model reads it and answers, with no tools involved.

Only the model changes, so the score says how well each one reads the programme and follows the answer format.

In [14]:
without_tools = {}

for name, agent in simple_agents.items():
    without_tools[name] = dataset.run_experiment(
        name=f"no-tools-{name.replace('/', '-')}",
        task=create_task(agent),
        evaluators=[finds_expected_talks],
        max_concurrency=1,
    )

langfuse.flush()
compare(without_tools)

request  \
experiment                                                              
openai/gpt-4.1-mini     0        What is on Friday morning about NLP?   
                        1     I want to learn about data engineering.   
                        2  Show me the visualisation talks on Friday.   
                        3                Which MLOps talks are there?   
                        4   I am interested in NLP talks on Thursday.   
google/gemini-2.5-flash 0        What is on Friday morning about NLP?   
                        1     I want to learn about data engineering.   
                        2  Show me the visualisation talks on Friday.   
                        3                Which MLOps talks are there?   
                        4   I am interested in NLP talks on Thursday.   

                           finds_expected_talks  
experiment                                       
openai/gpt-4.1-mini     0              1.000000  
                        1              1.000000  
                        2              1.000000  
                        3              1.000000  
                        4              0.666667  
google/gemini-2.5-flash 0              1.000000  
                        1              1.000000  
                        2              1.000000  
                        3              1.000000  
                        4              1.000000

## 8. Experiment 2: searching the programme and saving a file

The programme is not in the prompt any more. Each model has to call `search_talks` to find the talks, and `save_favorites` to write the file.

We score the answer and the file separately, so a model can get one right and the other wrong.

In [15]:
with_tools = {}

for name, agent in tool_agents.items():
    with_tools[name] = dataset.run_experiment(
        name=f"tools-{name.replace('/', '-')}",
        task=create_task(agent),
        evaluators=[finds_expected_talks, saves_expected_talks],
        max_concurrency=1,
    )

langfuse.flush()
compare(with_tools)

request  \
experiment                                                              
openai/gpt-4.1-mini     0        What is on Friday morning about NLP?   
                        1     I want to learn about data engineering.   
                        2  Show me the visualisation talks on Friday.   
                        3                Which MLOps talks are there?   
                        4   I am interested in NLP talks on Thursday.   
google/gemini-2.5-flash 0        What is on Friday morning about NLP?   
                        1     I want to learn about data engineering.   
                        2  Show me the visualisation talks on Friday.   
                        3                Which MLOps talks are there?   
                        4   I am interested in NLP talks on Thursday.   

                           finds_expected_talks  saves_expected_talks  
experiment                                                             
openai/gpt-4.1-mini     0                   1.0                  True  
                        1                   1.0                 False  
                        2                   1.0                  True  
                        3                   1.0                 False  
                        4                   1.0                  True  
google/gemini-2.5-flash 0                   1.0                 False  
                        1                   0.0                  True  
                        2                   1.0                 False  
                        3                   1.0                 False  
                        4                   1.0                 False

Open a trace in Langfuse. Each tool call is its own step, so you can see what the agent decided and in which order.